In [40]:
import pandas as pd
import numpy as np
import os
from datetime import timedelta

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
def get_data(filename):

    return pd.read_csv(os.path.join('..','data','raw',f'{filename}.csv'))

In [3]:
def calculate_distance(record):
    lat1 = np.radians( record['customer_latitude'])
    lng1 = np.radians(record['customer_longitude'])
    lat2 = np.radians(record['seller_latitude'])
    lng2 = np.radians(record['seller_longitude'])

    # Calucate the difference.
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    
    a = (np.sin(dlat/2)**2) + np.cos(lat1) * np.cos(lat2) * np.sin(dlng/2)**2

    c =  2 * np.asin(np.sqrt(a))

    r = 6371

    return np.round((c * r),0)
    

In [4]:
customers =get_data('customers')
sellers = get_data('sellers')
geolocation = get_data('geolocation')
orders = get_data('orders')
items = get_data('order_items')
products = get_data('products')
payments = get_data('order_payments')

category = get_data('product_category_name_translation')

In [5]:
#  Converting object to datetime data type.
dates_cols = ['order_purchase_timestamp','order_approved_at',
              'order_delivered_carrier_date','order_delivered_customer_date',
              'order_estimated_delivery_date']

for col in dates_cols:
    orders[col] = pd.to_datetime(orders[col])



In [6]:

# 
unique_geo = geolocation.groupby('geolocation_zip_code_prefix').agg(lat = ('geolocation_lat','median'),lng = ('geolocation_lng','median'))

cust_geo = customers.merge(unique_geo.rename(columns ={'lat':'customer_latitude','lng':'customer_longitude'}), 
                           left_on='customer_zip_code_prefix',right_on='geolocation_zip_code_prefix',how='left')


seller_geo = sellers.merge(unique_geo.rename(columns ={'lat':'seller_latitude','lng':'seller_longitude'}), 
                           left_on='seller_zip_code_prefix',right_on='geolocation_zip_code_prefix',how='left')

### Product

In [7]:
products['product_volume_cm3'] = products['product_length_cm'] * products['product_height_cm'] * products['product_width_cm'] 

category['product_category_name_english'] = category['product_category_name_english'].str.lower()
products_cat = products.merge(category, on='product_category_name',how='left')

### Item table

In [8]:
# working on items table
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])
items_full =  items.merge(seller_geo,on='seller_id',how='left').merge(products_cat[['product_volume_cm3','product_category_name_english','product_weight_g','product_id']],on='product_id',how='left')

In [9]:
items_full.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,seller_zip_code_prefix,seller_city,seller_state,seller_latitude,seller_longitude,product_volume_cm3,product_category_name_english,product_weight_g
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,27277,volta redonda,SP,-22.499217,-44.125997,3528.0,cool_stuff,650.0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,3471,sao paulo,SP,-23.566116,-46.519215,60000.0,pet_shop,30000.0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,37564,borda da mata,MG,-22.272310,-46.165743,14157.0,furniture_decor,3050.0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,14403,franca,SP,-20.554801,-47.387749,2400.0,perfumery,200.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,87900,loanda,PR,-22.930512,-53.136361,42000.0,garden_tools,3750.0


In [10]:
items_agg = items_full.groupby('order_id').agg(
    n_items = ('order_item_id','max'),
    seller_id = ('seller_id','first'),
    shipping_limit_date = ('shipping_limit_date','max'),
    total_price = ('price','sum'),
    total_freight_value = ('freight_value','sum'),
    seller_zip_code = ('seller_zip_code_prefix','first'),
    seller_state  = ('seller_state','first'),
    seller_latitude = ('seller_latitude','first'),
    seller_longitude = ('seller_longitude','first'),
    total_volume= ('product_volume_cm3','sum'),
    category = ('product_category_name_english','first'),
    total_product_weight_g = ('product_weight_g','sum'),
    n_unique_sellers = ('seller_id','nunique'),
    n_unique_states = ('seller_state','nunique'),
).reset_index()

### payment

In [11]:
payments_agg = payments.groupby('order_id').agg(payment_type = ('payment_type','first'),
                                 payment_installments=('payment_installments','max'),
                                 payment_value=('payment_value','sum')).reset_index()

In [12]:
df = (orders.merge(cust_geo,on='customer_id',how='left',validate='1:1')
    .merge(items_agg,on='order_id',how='left',validate='1:1')
    .merge(payments_agg,how='left',on='order_id',validate='1:1'))

df = df[df['order_status']=='delivered']

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96478 entries, 0 to 99440
Data columns (total 31 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96478 non-null  object        
 1   customer_id                    96478 non-null  object        
 2   order_status                   96478 non-null  object        
 3   order_purchase_timestamp       96478 non-null  datetime64[ns]
 4   order_approved_at              96464 non-null  datetime64[ns]
 5   order_delivered_carrier_date   96476 non-null  datetime64[ns]
 6   order_delivered_customer_date  96470 non-null  datetime64[ns]
 7   order_estimated_delivery_date  96478 non-null  datetime64[ns]
 8   customer_unique_id             96478 non-null  object        
 9   customer_zip_code_prefix       96478 non-null  int64         
 10  customer_city                  96478 non-null  object        
 11  customer_state      

### Split the data into train_test_split

In [14]:
split_point = df['order_purchase_timestamp'].quantile(0.80)
training_data = df[df['order_purchase_timestamp']<=split_point].copy()
test_data =  df[df['order_purchase_timestamp']>split_point].copy()

### Feature engineering

In [15]:
# target_feature.
df['delivery_days'] =  (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

In [16]:
zero_time = timedelta(0)
df['is_shipped_on_time'] =  np.where((df['shipping_limit_date'] - df['order_delivered_carrier_date'])>zero_time,1,0)

In [17]:
df['is_same_state'] = np.where(df['customer_state'] == df['seller_state'],1,0)

In [18]:
df['delivery_distance'] = df.apply(calculate_distance,axis=1)

In [19]:
df['freight_ratio'] = df['total_freight_value'] / (df['total_price'] + 1)


In [20]:
# ──  Weight per item (density proxy) ───────────────────────────────────────
df['weight_per_item'] = df['total_product_weight_g'] / (df['n_items'] + 1)
df['volume_per_item'] = df['total_volume'] / (df['n_items'] + 1)

In [21]:


# ── Distance x Weight interaction (shipping complexity) ───────────────────
df['distance_x_weight'] = df['delivery_distance'] * df['total_product_weight_g']
df['distance_x_items']  = df['delivery_distance'] * df['n_items']


In [22]:

# ── Cross-state multi-seller complexity ────────────────────────────────────
df['logistics_complexity'] = df['n_unique_sellers'] * df['n_unique_states']

###  Seller previous delivery on time rate and avg delivery days

In [23]:
seller_performance = df[['order_id','order_purchase_timestamp','order_delivered_customer_date','delivery_days','is_shipped_on_time','seller_id']].copy()

In [24]:
seller_performance.sort_values(['seller_id','order_purchase_timestamp'],inplace=True)

In [25]:
seller_previous_order_count = seller_performance.groupby('seller_id').cumcount()

In [26]:
seller_timely_delivery_avg = seller_performance.groupby('seller_id')['is_shipped_on_time'].transform(lambda x: x.expanding().mean().shift())

In [27]:
df['seller_previous_order_count'] = seller_previous_order_count
df['seller_timely_delivery_avg'] = seller_timely_delivery_avg

In [28]:
df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,is_same_state,delivery_distance,freight_ratio,weight_per_item,volume_per_item,distance_x_weight,distance_x_items,logistics_complexity,seller_previous_order_count,seller_timely_delivery_avg
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,28013,...,0,301.0,0.221870,325.0,1764.0,195650.0,301.0,1.0,83,0.86747
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,15775,...,1,589.0,0.082731,15000.0,30000.0,17670000.0,589.0,1.0,0,NaN
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,35661,...,1,313.0,0.089350,1525.0,7078.5,954650.0,313.0,1.0,6,1.00000
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,af861d436cfc08b2c2ddefd0ba074622,12952,...,1,295.0,0.914224,100.0,1200.0,59000.0,295.0,1.0,10,0.90000
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,64b576fb70d441e8f1b2d7d446e483c5,13226,...,0,646.0,0.090294,1875.0,21000.0,2422500.0,646.0,1.0,3,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,fffc94f6ce00a00581880bf54a75a037,b51593916b4b8e0d6f66f2ae24f2673d,delivered,2018-04-23 13:57:06,2018-04-25 04:11:01,2018-04-25 12:09:00,2018-05-10 22:56:40,2018-05-18,0c9aeda10a71f369396d0c04dce13a64,65077,...,0,2755.0,0.144224,5075.0,26700.0,27963250.0,2755.0,1.0,30,1.00000
99437,fffcd46ef2263f404302a634eb57f7eb,84c5d4fbaf120aae381fad077416eaa0,delivered,2018-07-14 10:26:46,2018-07-17 04:31:48,2018-07-17 08:05:00,2018-07-23 20:31:55,2018-08-01,0da9fe112eae0c74d3ba1fe16de0988b,81690,...,0,354.0,0.104074,4475.0,22230.0,3168300.0,354.0,1.0,10,1.00000
99438,fffce4705a9662cd70adb13d4a31832d,29309aa813182aaddc9b259e31b870e6,delivered,2017-10-23 17:07:56,2017-10-24 17:14:25,2017-10-26 15:13:14,2017-10-28 12:22:22,2017-11-10,cd79b407828f02fdbba457111c38e4c4,4039,...,0,339.0,0.167988,483.5,4788.0,327813.0,339.0,1.0,156,0.99359
99439,fffe18544ffabc95dfada21779c9644f,b5e6afd5a41800fdf401e0272ca74655,delivered,2017-08-14 23:02:59,2017-08-15 00:04:32,2017-08-15 19:02:53,2017-08-16 21:59:40,2017-08-25,eb803377c9315b564bdedad672039306,13289,...,1,73.0,0.153009,50.0,4000.0,7300.0,73.0,1.0,18,1.00000


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 96478 entries, 0 to 99440
Data columns (total 43 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96478 non-null  object        
 1   customer_id                    96478 non-null  object        
 2   order_status                   96478 non-null  object        
 3   order_purchase_timestamp       96478 non-null  datetime64[ns]
 4   order_approved_at              96464 non-null  datetime64[ns]
 5   order_delivered_carrier_date   96476 non-null  datetime64[ns]
 6   order_delivered_customer_date  96470 non-null  datetime64[ns]
 7   order_estimated_delivery_date  96478 non-null  datetime64[ns]
 8   customer_unique_id             96478 non-null  object        
 9   customer_zip_code_prefix       96478 non-null  int64         
 10  customer_city                  96478 non-null  object        
 11  customer_state      

In [30]:
corr_matrix = df[['customer_zip_code_prefix', 'customer_latitude', 'customer_longitude',
       'n_items', 'total_price', 'total_freight_value', 'seller_zip_code',
       'seller_latitude', 'seller_longitude', 'total_volume',
       'total_product_weight_g', 'n_unique_sellers', 'n_unique_states',
       'payment_installments', 'payment_value', 'delivery_days',
       'is_shipped_on_time', 'is_same_state', 'delivery_distance',
       'freight_ratio', 'weight_per_item', 'volume_per_item',
       'distance_x_weight', 'distance_x_items', 'logistics_complexity',
       'seller_previous_order_count', 'seller_timely_delivery_avg']].corr()

In [31]:
corr_matrix['delivery_days'].abs().sort_values(ascending=False)[1:16]

delivery_distance             0.393194
is_same_state                 0.361856
distance_x_items              0.326497
customer_zip_code_prefix      0.270996
customer_latitude             0.258151
is_shipped_on_time            0.231050
distance_x_weight             0.178928
total_freight_value           0.167110
seller_timely_delivery_avg    0.145667
customer_longitude            0.112376
freight_ratio                 0.087622
weight_per_item               0.079611
total_product_weight_g        0.071384
volume_per_item               0.071007
payment_value                 0.068959
Name: delivery_days, dtype: float64

In [38]:
selected_features = [
    # Core logistics
    'delivery_distance',
    'total_product_weight_g',
    'total_volume',
    'freight_ratio',
    'weight_per_item',
    'volume_per_item',

    # Order info
    'n_items',
    'total_price',
    'payment_value',
    'payment_installments',

    # Interactions
    'distance_x_weight',
    'logistics_complexity',

    # Seller behavior
    'seller_previous_order_count',
    'seller_timely_delivery_avg',

    # Structural
    'n_unique_sellers',
    'is_same_state',

    # prduct
    'category',
    'payment_type'
]

obj_cols = ['category','payment_type']
binary_cols = ['is_same_state']

In [41]:
# df[[target] + numeric_cols + obj_cols + extras]
master_data = df [selected_features + extras + [target]]

In [42]:
training_data , testing_data = train_test_split(master_data, test_size=0.2,random_state=42)

In [43]:
os.path.exists('../data/processed')

True

In [48]:
training_data.to_csv('../data/processed/training_data.csv',index=False)
testing_data.to_csv('../data/processed/testing_data.csv',index=False)